In [3]:
import base64
import json
import os
from contextlib import AsyncExitStack

from dotenv import load_dotenv
from IPython.display import Image, display
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from openai import OpenAI

load_dotenv(override=True)

API_KEY = os.getenv("GEMINI_API_KEY")
MODEL = "gemini-3.1-flash-lite-preview"  # test phase, lots of tokens


In [4]:
class MCPManager:
    def __init__(self, servers: dict[str, str]):
        self.servers = servers
        self.clients = {}
        self.tools = []
        self._stack = AsyncExitStack()

    async def __aenter__(self):
        for url in self.servers.values():
            read, write, session_id = await self._stack.enter_async_context(
                streamable_http_client(url)
            )
            session = await self._stack.enter_async_context(ClientSession(read, write))
            await session.initialize()

            tools_resp = await session.list_tools()
            for t in tools_resp.tools:
                self.clients[t.name] = session
                self.tools.append(
                    {
                        "type": "function",
                        "function": {
                            "name": t.name,
                            "description": t.description,
                            "parameters": t.inputSchema,
                        },
                    }
                )

        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self._stack.aclose()

    async def call_tool(self, name: str, args: dict) -> str:
        result = await self.clients[name].call_tool(name, arguments=args)
        return result.content[0].text


async def make_llm_request_mcp(prompt: str) -> str:
    mcp_servers = {
        "visualization": "http://localhost:8003/mcp",
    }

    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    async with MCPManager(mcp_servers) as mcp:
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant. Use tools if you need to.",
            },
            {"role": "user", "content": prompt},
        ]

        for _ in range(10):
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=mcp.tools,
                tool_choice="auto",
                max_completion_tokens=1000,
            )

            resp_message = response.choices[0].message
            if not resp_message.tool_calls:
                return resp_message.content

            messages.append(
                {k: v for k, v in resp_message.model_dump().items() if v is not None}
            )
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                print(f"Executing tool '{func_name}' with args:")
                print(json.dumps(func_args, indent=2))

                func_result = await mcp.call_tool(func_name, func_args)

                if func_name == "line_plot":
                    display(Image(data=base64.b64decode(func_result)))
                    tool_content = "Plot generated and displayed successfully."
                else:
                    tool_content = func_result

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": tool_content,
                    }
                )


In [5]:
prompt = "Plot sin(x) using only 10 discrete x points."
response = await make_llm_request_mcp(prompt)
print("Response:", response)

print()

prompt = "Plot x^2 for these x values: [1, 2, 3, 4, 5]."
response = await make_llm_request_mcp(prompt)
print("Response:", response)


CancelledError: Cancelled via cancel scope 115cbb050